In [1]:
import os
import sys
import gc
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

ModuleNotFoundError: No module named 'lightgbm'

In [ ]:


# 警告の非表示設定
warnings.filterwarnings('ignore')

print(f"LightGBM Version: {lgb.__version__}")


# ==========================================
# Config / Configuration Settings
# ==========================================
class Config:
    SEASON = "s6e8"
    TRAIN_PATH = f"/kaggle/input/competitions/playground-series-{SEASON}/train.csv"
    TEST_PATH = f"/kaggle/input/competitions/playground-series-{SEASON}/test.csv"
    SUB_PATH = f"/kaggle/input/competitions/playground-series-{SEASON}/sample_submission.csv"

    MODEL_NAME = "LightGBM"
    TARGET = "addicted_label"
    ID_COL = "id"
    N_SPLITS = 5
    RANDOM_SEED = 42


# ==========================================
# Optimized Feature Engineering Protocol
# ==========================================
def create_optimal_features(df):
    """
    主要な比率と滑らかな変換に焦点を当てたロバストな特徴量エンジニアリング
    """
    df = df.copy()
    eps = 1e-3

    # 1. Total Missing Counts
    df['total_missing_count'] = df.isna().sum(axis=1)

    # 2. Awake & Extreme Screen Ratios
    if 'sleep_hours' in df.columns:
        df['awake_hours_calc'] = (24.0 - df['sleep_hours']).clip(lower=1.0)
        if 'daily_screen_time_hours' in df.columns:
            df['screen_ratio_of_awake'] = df['daily_screen_time_hours'] / (df['awake_hours_calc'] + eps)
    else:
        df['awake_hours_calc'] = 16.0

    track_cols = [c for c in ['social_media_hours', 'gaming_hours', 'work_study_hours'] if c in df.columns]
    if len(track_cols) == 3:
        df['tracked_sum_calc'] = df[track_cols].sum(axis=1, skipna=False)
    else:
        df['tracked_sum_calc'] = np.nan

    # 3. Untracked and Discrepancy Features
    if 'daily_screen_time_hours' in df.columns and 'tracked_sum_calc' in df.columns:
        df['untracked_time_calc'] = (df['daily_screen_time_hours'] - df['tracked_sum_calc']).clip(lower=0.0)
        df['screen_time_discrepancy'] = df['daily_screen_time_hours'] - df['tracked_sum_calc']
        df['untracked_ratio_to_awake'] = df['untracked_time_calc'] / (df['awake_hours_calc'] + eps)
        df['untracked_ratio_to_screen'] = df['untracked_time_calc'] / (df['daily_screen_time_hours'] + eps)
        df['has_untracked_gap_smooth'] = (df['untracked_time_calc'] > 0.25).astype(float)

    # 4. Missing Flags for Key Variables
    for col in ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'notifications_per_day']:
        if col in df.columns:
            df[f'is_missing_{col}'] = df[col].isna().astype(float)

    # 5. Gaussian Kernel Smooth Mid-Zone
    if 'daily_screen_time_hours' in df.columns:
        df['is_mid_screen_time_smooth'] = np.exp(-0.5 * ((df['daily_screen_time_hours'] - 6.25) / 1.75) ** 2)

    # 6. Leisure & Work Ratios
    if 'work_study_hours' in df.columns and 'social_media_hours' in df.columns and 'gaming_hours' in df.columns:
        leisure_calc = df['social_media_hours'].fillna(0) + df['gaming_hours'].fillna(0)
        df['work_to_leisure_ratio_smooth'] = (df['work_study_hours'] + 0.1) / (leisure_calc + 0.1)
        
        if 'daily_screen_time_hours' in df.columns:
            df['work_leisure_balance_index'] = (df['work_study_hours'] * leisure_calc) / ((df['daily_screen_time_hours'] ** 2) + eps)
            df['leisure_ratio_of_screen'] = leisure_calc / (df['daily_screen_time_hours'] + eps)

    if 'notifications_per_day' in df.columns and 'app_opens_per_day' in df.columns:
        df['notif_per_open_smooth'] = (df['notifications_per_day'] + 1.0) / (df['app_opens_per_day'] + 1.0)
        if 'daily_screen_time_hours' in df.columns:
            df['notif_per_screen_hour'] = df['notifications_per_day'] / (df['daily_screen_time_hours'] + eps)

    # 7. Interactions
    if 'daily_screen_time_hours' in df.columns:
        if 'is_mid_screen_time_smooth' in df.columns and 'social_media_hours' in df.columns:
            df["mid_zone_social_ratio"] = df["is_mid_screen_time_smooth"] * (df["social_media_hours"] / (df["daily_screen_time_hours"] + eps))
        if 'is_mid_screen_time_smooth' in df.columns and 'gaming_hours' in df.columns:
            df["mid_zone_gaming_ratio"] = df["is_mid_screen_time_smooth"] * (df["gaming_hours"] / (df["daily_screen_time_hours"] + eps))

    # 8. Weekend vs Daily Screen Time
    if 'weekend_screen_time' in df.columns and 'daily_screen_time_hours' in df.columns:
        df["weekend_to_daily_ratio"] = df["weekend_screen_time"] / (df["daily_screen_time_hours"] + eps)
        df["weekend_daily_diff"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
        df["weekend_daily_log_ratio"] = np.log1p(df["weekend_screen_time"]) - np.log1p(df["daily_screen_time_hours"])

    # 9. Direct Category Ratios
    if 'work_study_hours' in df.columns and 'social_media_hours' in df.columns:
        df["work_to_social_ratio"] = df["work_study_hours"] / (df["social_media_hours"] + eps)
    if 'work_study_hours' in df.columns and 'gaming_hours' in df.columns:
        df["work_to_gaming_ratio"] = df["work_study_hours"] / (df["gaming_hours"] + eps)
    if 'social_media_hours' in df.columns and 'gaming_hours' in df.columns:
        df["social_to_gaming_ratio"] = df["social_media_hours"] / (df["gaming_hours"] + eps)

    # Cleanup temporary columns
    drop_temp = ['awake_hours_calc', 'tracked_sum_calc', 'untracked_time_calc']
    df.drop(columns=[c for c in drop_temp if c in df.columns], inplace=True)

    return df


# ==========================================
# Main Processing & Model Training
# ==========================================
def main():
    print("Loading datasets...")
    train_raw = pd.read_csv(Config.TRAIN_PATH)
    test_raw = pd.read_csv(Config.TEST_PATH)
    sub_df = pd.read_csv(Config.SUB_PATH)

    y_train = train_raw[Config.TARGET].values
    X_train_raw = train_raw.drop(columns=[Config.ID_COL, Config.TARGET])
    X_test_raw = test_raw.drop(columns=[Config.ID_COL])

    print("Applying feature engineering...")
    X_train_feat = create_optimal_features(X_train_raw)
    X_test_feat = create_optimal_features(X_test_raw)

    cat_cols = [c for c in X_train_feat.columns if X_train_feat[c].dtype == 'object']

    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'n_estimators': 8000,
        'learning_rate': 0.018,
        'num_leaves': 45,
        'max_depth': 7,
        'min_child_samples': 200,
        'subsample': 0.8,
        'subsample_freq': 1,
        'colsample_bytree': 0.70,
        'reg_alpha': 3.0,
        'reg_lambda': 5.0,
        'random_state': Config.RANDOM_SEED,
        'verbose': -1,
        'n_jobs': -1
    }

    oof_preds = np.zeros(len(train_raw))
    test_preds = np.zeros(len(test_raw))

    skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.RANDOM_SEED)

    print("\n==========================================")
    print(f" Running Strictly Validated {Config.MODEL_NAME}")
    print("==========================================")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_feat, y_train)):
        X_tr = X_train_feat.iloc[train_idx].copy()
        y_tr = y_train[train_idx]
        X_va = X_train_feat.iloc[val_idx].copy()
        y_va = y_train[val_idx]
        X_te = X_test_feat.copy()

        # Fold-Safe Screentime Distance from Median
        if 'daily_screen_time_hours' in X_tr.columns:
            median_screen = X_tr["daily_screen_time_hours"].median()
            for df_curr in [X_tr, X_va, X_te]:
                df_curr["screentime_dist_from_median"] = np.abs(df_curr["daily_screen_time_hours"] - median_screen)

        # Fold-Safe Target Encoding
        global_mean = float(y_tr.mean())
        smooth_weight = 10.0

        for ccol in cat_cols:
            tr_cat = X_tr[ccol].astype(str)
            va_cat = X_va[ccol].astype(str)
            te_cat = X_te[ccol].astype(str)

            temp_df = pd.DataFrame({'cat': tr_cat, 'target': y_tr})
            stats = temp_df.groupby('cat')['target'].agg(['count', 'mean'])
            te_dict = ((stats['count'] * stats['mean'] + smooth_weight * global_mean) / (stats['count'] + smooth_weight)).to_dict()

            X_tr[f'{ccol}_te'] = tr_cat.map(te_dict).fillna(global_mean).astype(float)
            X_va[f'{ccol}_te'] = va_cat.map(te_dict).fillna(global_mean).astype(float)
            X_te[f'{ccol}_te'] = te_cat.map(te_dict).fillna(global_mean).astype(float)

            # 不要となった元のカテゴリ列を削除して多重共線性を防止
            X_tr.drop(columns=[ccol], inplace=True)
            X_va.drop(columns=[ccol], inplace=True)
            X_te.drop(columns=[ccol], inplace=True)

        # Fold-Safe Age Group Statistics
        if 'age' in X_tr.columns:
            eps = 1e-3
            _, age_bins = pd.qcut(X_tr['age'].dropna(), q=6, retbins=True, duplicates='drop')
            age_bins[0] = -np.inf
            age_bins[-1] = np.inf

            for df_curr in [X_tr, X_va, X_te]:
                df_curr['age_group'] = pd.cut(df_curr['age'], bins=age_bins, labels=False, include_lowest=True).fillna(-1)

            target_agg_cols = [c for c in ['daily_screen_time_hours', 'notifications_per_day'] if c in X_tr.columns]
            for col in target_agg_cols:
                age_means = X_tr.groupby('age_group')[col].mean()
                age_stds = X_tr.groupby('age_group')[col].std()

                for df_curr in [X_tr, X_va, X_te]:
                    mean_val = df_curr['age_group'].map(age_means)
                    std_val = df_curr['age_group'].map(age_stds)
                    df_curr[f'{col}_diff_from_age_avg'] = df_curr[col] - mean_val
                    df_curr[f'{col}_zscore_in_age'] = (df_curr[col] - mean_val) / (std_val + eps)

            for df_curr in [X_tr, X_va, X_te]:
                df_curr.drop(columns=['age_group'], inplace=True)

        # Model Training
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)]
        )

        val_pred = model.predict_proba(X_va)[:, 1]
        test_pred = model.predict_proba(X_te)[:, 1]

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / Config.N_SPLITS

        fold_auc = roc_auc_score(y_va, val_pred)
        print(f" Fold {fold + 1} AUC: {fold_auc:.5f} (Best Iteration: {model.best_iteration_})")

        del X_tr, X_va, X_te, model
        gc.collect()

    overall_auc = roc_auc_score(y_train, oof_preds)
    print(f"\n==> {Config.MODEL_NAME} Overall OOF AUC: {overall_auc:.5f}")

    # npyファイル形式での保存（必須仕様）
    oof_filename = f"oof_preds_{Config.MODEL_NAME}_0.npy"
    test_filename = f"test_preds_{Config.MODEL_NAME}_0.npy"
    np.save(oof_filename, oof_preds)
    np.save(test_filename, test_preds)
    print(f"Saved: {oof_filename}, {test_filename}")

    # 提出用 submission.csv の保存
    final_sub = sub_df.copy()
    final_sub[Config.TARGET] = test_preds
    final_sub.to_csv("submission.csv", index=False)
    print("Saved: submission.csv successfully!")


if __name__ == "__main__":
    main()